# 09 · Yapay Sinir Ağları
Katmanlar halinde bağlı nöronlar: her nöron $a = f(w^Tx + b)$ hesaplar ($f$: ReLU, sigmoid gibi aktivasyon). Ağırlıklar **geri yayılım (backpropagation)** + **gradyan inişi** ile öğrenilir.

| Ağ türü | Veri türü |
|---|---|
| **MLP (Dense)** | Tablo verisi |
| **CNN** | Görüntü |
| **RNN / LSTM / GRU** | Sıralı veri, zaman serisi, metin |
| **Transformer** | Metin (BERT, GPT), görüntü, ses |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, recall_score,
                             classification_report, ConfusionMatrixDisplay, RocCurveDisplay)

from sklearn.neural_network import MLPClassifier

In [ ]:
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)   # stratify: sınıf oranlarını koru

sonuclar = {}
def degerlendir(model, ad):
    model.fit(X_train, y_train)
    p = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    sonuclar[ad] = {"Accuracy": accuracy_score(y_test, p), "F1": f1_score(y_test, p),
                    "Recall": recall_score(y_test, p),
                    "ROC-AUC": roc_auc_score(y_test, proba) if proba is not None else np.nan}
    print(f"{ad}: " + "  ".join(f"{k}={v:.3f}" for k, v in sonuclar[ad].items()))
    return model

## 1. scikit-learn MLP (basit)

In [ ]:
mlp = degerlendir(Pipeline([("s", StandardScaler()),
                            ("m", MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000,
                                                early_stopping=True, random_state=42))]), "MLP (sklearn)")
plt.figure(figsize=(7, 3.5)); plt.plot(mlp.named_steps["m"].loss_curve_, color="tab:blue")
plt.title("MLP – eğitim kaybı"); plt.xlabel("iterasyon"); plt.show()

**Yorum:** Kayıp hızla düşüp sabitleniyor. `early_stopping=True`, validation skoru iyileşmeyi bıraktığında eğitimi durdurarak overfitting'i önler.

## 2. Keras / TensorFlow

In [ ]:
import tensorflow as tf
tf.random.set_seed(42)

scaler = StandardScaler().fit(X_train)                    # sadece train'e fit
Xtr_s, Xte_s = scaler.transform(X_train), scaler.transform(X_test)

nn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(Xtr_s.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),                         # overfitting'e karşı
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),       # ikili sınıflandırma
])
nn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
nn.summary()

In [ ]:
gecmis = nn.fit(Xtr_s, y_train, validation_split=0.2, epochs=200, batch_size=32, verbose=0,
                callbacks=[tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)])

h = pd.DataFrame(gecmis.history)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
h[["loss", "val_loss"]].plot(ax=ax[0], title="Kayıp", color=["tab:blue", "tab:orange"])
h[["accuracy", "val_accuracy"]].plot(ax=ax[1], title="Doğruluk", color=["tab:blue", "tab:orange"])
plt.tight_layout(); plt.show()
print("Durduğu epoch:", len(h))

**Yorum (öğrenme eğrisi):** `loss` düşmeye devam ederken `val_loss` yükselmeye başlarsa overfitting başlamıştır. EarlyStopping bu noktada eğitimi durdurup en iyi ağırlıkları geri yükler.

In [ ]:
proba = nn.predict(Xte_s, verbose=0).ravel()
p = (proba > 0.5).astype(int)
sonuclar["Keras NN"] = {"Accuracy": accuracy_score(y_test, p), "F1": f1_score(y_test, p),
                        "Recall": recall_score(y_test, p), "ROC-AUC": roc_auc_score(y_test, proba)}
pd.DataFrame(sonuclar).T.round(3)

## Özet
### Data Analysis Key Findings
- Hem sklearn MLP hem Keras ağı bu veri setinde ≈%97+ doğruluğa ulaşır — lojistik regresyonla benzer.
- Küçük tablo verilerinde sinir ağları genellikle ağaç ensemble'larına veya lineer modellere üstünlük sağlamaz; asıl güçleri görüntü, metin, ses gibi yapılandırılmamış veridedir.
### Insights or Next Steps
- Görüntü için CNN (`Conv2D`), metin/zaman serisi için LSTM veya Transformer katmanları deneyin.